In [1]:
import pandas as pd
df = pd.read_csv("../../data/S12_PD_Diaz_DatasetIntegrado.csv")

# Ejercicio Semana 13

## Ingeniería de características

### Features
- dias_desde_ultima_compra = fecha_de_referencia - fecha_ultima_compra
- es_cliente_recurrente = numero_compras > 1
- resumen_cliente = suma de monto, conteo de id_compra, promedio de monto
- compras_ultimos_30d = suma de id_compra desde hace 30 días
- score_R = Es la feature de Recencia. Mide qué tan reciente fue la última compra del cliente (recibe un puntaje más alto si compró hace poco).
- score_F = Es la feature de Frecuencia. Mide qué tan seguido compra el cliente.
- score_M = Es la feature de Monetización. Mide cuánto dinero gasta en total.
- rfm_score = score_R + score_F + score_M
- es_temporada_alta = 1 si ["fecha_pedido"].dt.month.isin([11, 12])
- categoria_favorita = conteo de compras por categia_producto, oredenar de mayor a menor y tomar el de conteo más alto
- canal_principal = conteo de compras por canal_compra, oredenar de mayor a menor y tomar el de conteo más alto

### Día última compra

In [3]:
import pandas as pd

hoy = pd.Timestamp("2026-10-19")  # fecha de referencia del corte de análisis
df["dias_desde_ultima_compra"] = (hoy - df["fecha_ultima_compra"]).dt.days

df["dias_desde_ultima_compra"].describe()

KeyError: 'fecha_ultima_compra'

### Cliente recurrente

In [ ]:
df["es_cliente_recurrente"] = (df["numero_compras"] > 1)

df["es_cliente_recurrente"].value_counts()

### Resumen cliente

In [ ]:
# Resumen cliente
resumen_cliente = (
    df.groupby("id_cliente")
      .agg(
          total_gastado=("monto", "sum"),
          numero_compras=("id_compra", "count"),
          gasto_promedio=("monto", "mean"),
      )
      .reset_index()
)
df["resumen_cliente"].describe()

### Actividad reciente

In [ ]:
# Actividad reciente

hace_30_dias = hoy - pd.Timedelta(days=30)
compras_recientes = df[df["fecha_pedido"] >= hace_30_dias]

compras_ultimos_30d = (
    compras_recientes.groupby("id_cliente")["id_compra"]
    .count()
    .rename("compras_ultimos_30_dias")
)
df["compras_ultimos_30_dias"].describe()

### RFM

In [ ]:
# RFM

# Se asume que resumen_cliente ya tiene: dias_desde_ultima_compra,
# numero_compras y total_gastado (de las lecciones anteriores).

resumen_cliente["score_R"] = pd.qcut(
    resumen_cliente["dias_desde_ultima_compra"], q=4, labels=[4, 3, 2, 1]
)
resumen_cliente["score_F"] = pd.qcut(
    resumen_cliente["numero_compras"].rank(method="first"), q=4, labels=[1, 2, 3, 4]
)
resumen_cliente["score_M"] = pd.qcut(
    resumen_cliente["total_gastado"], q=4, labels=[1, 2, 3, 4]
)
resumen_cliente["rfm_score"] = (
    resumen_cliente["score_R"].astype(int)
    + resumen_cliente["score_F"].astype(int)
    + resumen_cliente["score_M"].astype(int)
)
df["rfm_score"].describe()

### Es temporada alta

In [ ]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])
# Bandera a nivel de transacción indicando si el mes es Noviembre (11) o Diciembre (12)
df["es_temporada_alta"] = df["fecha_pedido"].dt.month.isin([11, 12]).astype(int)
df["es_temporada_alta"].value_counts()

### Categoría favorita

In [ ]:
cat_conteo = (
    df.groupby(["id_cliente", "categoria_producto"])
    .size()
    .reset_index(name="conteo_compras")
)

# Ordenar de mayor a menor y quedarse con la primera de cada cliente (la favorita)
categoria_favorita = (
    cat_conteo.sort_values(["id_cliente", "conteo_compras"], ascending=[True, False])
    .drop_duplicates(subset=["id_cliente"])
    [["id_cliente", "categoria_producto"]]
    .rename(columns={"categoria_producto": "categoria_favorita"})
)
df["categoria_favorita"].value_counts()

### Canal principal

In [ ]:
# Contar transacciones por cliente y canal
canal_conteo = (
    df.groupby(["id_cliente", "canal_compra"])
    .size()
    .reset_index(name="conteo_canal")
)

# Seleccionar el canal con mayor frecuencia por cliente
canal_principal = (
    canal_conteo.sort_values(["id_cliente", "conteo_canal"], ascending=[True, False])
    .drop_duplicates(subset=["id_cliente"])
    [["id_cliente", "canal_compra"]]
    .rename(columns={"canal_compra": "canal_principal"})
)
df["canal_principal"].value_counts()

In [ ]:
df.to_csv("novamarket_con_features.csv", index=False)